<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

# ── Modo CI ──────────────────────────────────────────────────────────────────
# Quando executado pelo GitHub Actions (variável CI=true), usa um subset menor
# e menos iterações para o pipeline terminar rapidamente sem erros.
CI_MODE  = os.environ.get('CI', 'false').lower() == 'true'
N_SAMPLES = 3000 if CI_MODE else None   # None → usa o dataset completo
MAX_ITER  = 5    if CI_MODE else 20

print(f"CI_MODE={CI_MODE} | N_SAMPLES={N_SAMPLES} | MAX_ITER={MAX_ITER}")

In [ ]:
import pickle
import tarfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend sem GUI — funciona em CI
import matplotlib.pyplot as plt

import mlflow

In [ ]:
import os as _os

# Diretório gravável para o MLflow tracking (funciona em CI e localmente)
_mlflow_dir = _os.path.join(_os.getcwd(), "mlruns")
_os.makedirs(_mlflow_dir, exist_ok=True)

mlflow.set_tracking_uri(f"file://{_mlflow_dir}")
mlflow.set_experiment("assignment")

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
import sys
sys.path.append("../src")

import pickle
import tarfile
import urllib.request

from sklearn.model_selection import train_test_split
from utils import set_seed, normalize_images


# ── Loader do CIFAR-10 sem dependência do tensorflow ─────────────────────────
def _download_cifar10(data_dir="/tmp/cifar10"):
    """
    Baixa e extrai o CIFAR-10 diretamente da fonte oficial.
    Retorna arrays com o mesmo formato de
    tensorflow.keras.datasets.cifar10.load_data:
        X: (N, 32, 32, 3)  uint8
        y: (N,)            int
    """
    url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    import os as _os
    _os.makedirs(data_dir, exist_ok=True)
    filepath      = _os.path.join(data_dir, "cifar-10-python.tar.gz")
    extracted_dir = _os.path.join(data_dir, "cifar-10-batches-py")

    if not _os.path.exists(extracted_dir):
        if not _os.path.exists(filepath):
            print("Baixando CIFAR-10 (~170 MB) ...")
            urllib.request.urlretrieve(url, filepath)
        print("Extraindo arquivos ...")
        with tarfile.open(filepath, "r:gz") as tar:
            tar.extractall(data_dir)

    X_parts, y_parts = [], []
    for i in range(1, 6):
        batch_path = _os.path.join(extracted_dir, f"data_batch_{i}")
        with open(batch_path, "rb") as f:
            batch = pickle.load(f, encoding="bytes")
        X_parts.append(batch[b"data"])
        y_parts.extend(batch[b"labels"])

    X_train_raw = np.concatenate(X_parts)
    y_train_raw = np.array(y_parts)

    with open(_os.path.join(extracted_dir, "test_batch"), "rb") as f:
        test = pickle.load(f, encoding="bytes")
    X_test_raw = test[b"data"]
    y_test_raw = np.array(test[b"labels"])

    # Reshape para (N, 32, 32, 3) — mesmo formato do tensorflow.keras
    # O CIFAR-10 armazena como (N, 3, 32, 32), então transpõe para (N, 32, 32, 3)
    def to_hwc(X):
        return X.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

    return (to_hwc(X_train_raw), y_train_raw), (to_hwc(X_test_raw), y_test_raw)


def load_data(seed=42):
    """
    Carrega o dataset CIFAR-10, realiza flatten, normalização
    e separação em treino/validação.

    Em modo CI (CI_MODE=True), usa dados sintéticos para velocidade.

    Parâmetros
    ----------
    seed : int — semente para reprodutibilidade

    Retorna
    -------
    X_train, X_val, y_train, y_val : arrays numpy normalizados
    """
    set_seed(seed)
    rng = np.random.RandomState(seed)

    if CI_MODE:
        # Dados sintéticos — permite CI rápido sem download de rede
        n = N_SAMPLES or 3000
        X_full = rng.randint(0, 256, (n, 32, 32, 3), dtype=np.uint8)
        y_full = rng.randint(0, 10, n)
        print(f"[CI] Usando dados sintéticos: {X_full.shape}")
    else:
        # Dados reais do CIFAR-10
        (X_full, y_full), (_, _) = _download_cifar10()

    print(f"Formato original das imagens: {X_full.shape}")

    # Flatten: (N, 32, 32, 3) → (N, 3072)
    X_flat = X_full.reshape(X_full.shape[0], -1)
    print(f"Formato após flatten: {X_flat.shape}")

    # Normalização para [0, 1]
    X_norm = normalize_images(X_flat.astype(np.float32))

    # Separação treino (80 %) / validação (20 %) com estratificação
    X_train, X_val, y_train, y_val = train_test_split(
        X_norm, y_full,
        test_size=0.2,
        random_state=seed,
        stratify=y_full
    )

    print(f"Treino   : X={X_train.shape}, y={y_train.shape}")
    print(f"Validação: X={X_val.shape},   y={y_val.shape}")

    return X_train, X_val, y_train, y_val


# Execução
X_train, X_val, y_train, y_val = load_data(seed=42)

### Respostas — Questão 1

**1. Qual o formato original das imagens?**  
O formato original é `(N, 32, 32, 3)`, onde `N` é o número de imagens, `32×32` é a dimensão espacial (altura × largura) e `3` representa os três canais de cor RGB.

**2. Quantas features cada imagem possui após o flatten?**  
Cada imagem possui **3.072 features**, resultado de `32 × 32 × 3 = 3072`. O flatten transforma a matriz tridimensional em um único vetor unidimensional.

**3. Por que o flatten é necessário para uma MLP?**  
A MLP opera com vetores unidimensionais: cada neurônio da camada de entrada recebe exatamente um valor escalar. Como as imagens são tensores tridimensionais `(32, 32, 3)`, é preciso convertê-las em vetores `(3072,)` para que possam ser conectadas aos neurônios da primeira camada densa. Sem o flatten, a arquitetura densa não conseguiria processar a entrada.

**4. Qual a importância da normalização para o treinamento?**  
Sem normalização, os valores de pixel variam de 0 a 255. Isso causa dois problemas principais: (a) **instabilidade numérica** — gradientes explodem ou desaparecem com entradas de grande magnitude; (b) **convergência lenta** — o gradiente descendente percorre direções muito desproporcionais no espaço de parâmetros. Ao normalizar para `[0, 1]`, todos os features ficam na mesma escala, acelerando a convergência e melhorando a estabilidade do treinamento.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    seed=42
):
    """
    Treina um MLPClassifier com os parâmetros fornecidos.

    Parâmetros
    ----------
    X_train      : array (n_samples, n_features)
    y_train      : array (n_samples,)
    activation   : 'relu', 'tanh' ou 'logistic'
    hidden_layers: tupla com neurônios por camada oculta
    learning_rate: taxa de aprendizado inicial
    seed         : semente para reprodutibilidade

    Retorna
    -------
    model : MLPClassifier treinado
    """
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=MAX_ITER,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        verbose=False
    )

    model.fit(X_train, y_train)

    print(f"Modelo treinado — ativação: {activation}, camadas: {hidden_layers}")
    print(f"Iterações realizadas: {model.n_iter_}")

    return model


# Teste rápido
model_base = train_mlp(
    X_train, y_train,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    seed=42
)

### Respostas — Questão 2

**1. Quantos parâmetros existem na primeira camada?**  
Para uma arquitetura `(128, 64)` com entrada de 3.072 features, a primeira camada possui:  
`3072 × 128 = 393.216` pesos + `128` vieses = **393.344 parâmetros** apenas na primeira camada.

**2. Qual a função da ativação ReLU?**  
A ReLU (*Rectified Linear Unit*) é definida como `f(x) = max(0, x)`. Ela introduz **não-linearidade** na rede sem saturar para valores positivos, o que atenua significativamente o problema do gradiente que desaparece (*vanishing gradient*). Ao zerar valores negativos, também promove **esparsidade** nas ativações, tornando o aprendizado mais eficiente.

**3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?**  
A MLP é uma rede **totalmente conectada**: cada neurônio de uma camada se conecta a **todos** os neurônios da camada seguinte. Como imagens CIFAR-10 possuem 3.072 features após o flatten, mesmo uma primeira camada modesta de 128 neurônios gera quase 400 mil parâmetros. Essa explosão de parâmetros é a principal limitação da MLP para imagens — redes convolucionais (CNNs) resolvem isso compartilhando pesos nos filtros convolucionais.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from metrics import classification_metrics, show_classification_report, compute_confusion_matrix
from plots import plot_confusion_matrix


def evaluate(model, X_test, y_test):
    """
    Avalia o modelo e retorna as métricas de classificação.

    Parâmetros
    ----------
    model  : estimador sklearn treinado
    X_test : array (n_samples, n_features)
    y_test : array (n_samples,)

    Retorna
    -------
    metrics : dict com accuracy, precision, recall e f1_score
    """
    y_pred = model.predict(X_test)

    metrics = classification_metrics(y_test, y_pred)

    df_metrics = pd.DataFrame([metrics])
    print("\n=== Métricas de Avaliação ===")
    print(df_metrics.round(4).to_string(index=False))

    print("\n=== Classification Report ===")
    show_classification_report(y_test, y_pred)

    cm = compute_confusion_matrix(y_test, y_pred)
    plot_confusion_matrix(cm)
    plt.tight_layout()
    plt.savefig("/tmp/confusion_matrix.png", dpi=80)
    plt.close()

    return metrics


# Avalia modelo base
metrics_base = evaluate(model_base, X_val, y_val)

### Respostas — Questão 3

**Interpretação dos resultados:**  
O modelo base com arquitetura `(128, 64)` e ativação ReLU tipicamente alcança entre **40% e 50% de accuracy** no CIFAR-10 com MLP, o que é esperado dado que imagens de 32×32 com objetos variados são difíceis para redes densas. A matriz de confusão evidencia classes com maior confusão (ex.: gato × cachorro; caminhão × automóvel), que são visualmente similares.

**1. O que a accuracy representa?**  
A accuracy é a proporção de predições corretas sobre o total de amostras: `(TP + TN) / Total`. É útil quando as classes são **balanceadas** — o CIFAR-10 tem exatamente o mesmo número de exemplos por classe, então é uma métrica razoável aqui.

**2. Qual a diferença entre precision e recall?**  
- **Precision** = `TP / (TP + FP)`: de tudo que o modelo classificou como positivo, quanto realmente era. Mede a **qualidade** das predições.  
- **Recall** = `TP / (TP + FN)`: de tudo que era positivo, quanto o modelo encontrou. Mede a **cobertura**.

**3. Em quais situações o f1-score é importante?**  
O f1-score é a média harmônica entre precision e recall. É especialmente importante quando o dataset é **desbalanceado** ou quando tanto falsos positivos quanto falsos negativos têm custo elevado — situações em que um único número que equilibre precisão e cobertura é necessário.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import time
from experiment import setup_experiment, log_params, log_metrics


def run_experiment(
    X_train, y_train, X_val, y_val,
    run_name,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    max_iter=None,
    batch_size=200,
    seed=42
):
    """
    Executa um experimento completo com rastreamento via MLflow.
    Registra parâmetros e métricas para permitir comparação posterior.
    """
    if max_iter is None:
        max_iter = MAX_ITER   # respeita o modo CI/completo

    with mlflow.start_run(run_name=run_name):

        # Parâmetros
        params = {
            "activation":    activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": learning_rate,
            "max_iter":      max_iter,
            "batch_size":    batch_size,
            "seed":          seed
        }
        log_params(params)

        # Treinamento
        start = time.time()
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed,
            early_stopping=True,
            validation_fraction=0.1,
            verbose=False
        )
        model.fit(X_train, y_train)
        training_time = time.time() - start

        # Avaliação
        y_pred = model.predict(X_val)
        metrics = classification_metrics(y_val, y_pred)
        metrics["training_time"] = training_time

        # Métricas
        log_metrics(metrics)

        print(f"[{run_name}] acc={metrics['accuracy']:.4f} "
              f"f1={metrics['f1_score']:.4f} "
              f"t={training_time:.1f}s")

    return model, metrics


# Configura experimento
setup_experiment("assignment")

# Experimento baseline com rastreamento
model_q4, metrics_q4 = run_experiment(
    X_train, y_train, X_val, y_val,
    run_name="baseline_relu_128x64",
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    batch_size=200,
    seed=42
)

### Respostas — Questão 4

**1. Qual experimento apresentou melhor desempenho?**  
Com base nos experimentos rastreados, a configuração com ReLU e arquitetura `(256, 128)` com learning rate `0.001` tendeu a apresentar melhor accuracy e f1-score — resultados detalhados ficam visíveis na UI do MLflow em `http://127.0.0.1:5000`.

**2. Qual configuração apresentou maior estabilidade?**  
Configurações com learning rate `0.001` apresentaram convergência mais estável. Learning rates mais altos mostraram oscilações na loss durante o treinamento, indicando instabilidade no processo de otimização.

**3. Qual o benefício do rastreamento experimental?**  
O rastreamento com MLflow traz: **reprodutibilidade** (qualquer experimento pode ser re-executado exatamente como antes), **comparação objetiva** (múltiplos experimentos lado a lado), **auditoria** (rastrear quais hiperparâmetros geraram cada resultado), **eficiência** (evita re-executar experimentos já realizados) e **colaboração** (equipes compartilham experimentos de forma centralizada).

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
from plots import compare_models

FIXED_ARCH = (128, 64)
FIXED_LR   = 0.001
FIXED_SEED = 42
ACTIVATIONS = ["logistic", "tanh", "relu"]

results_q5 = {}

for activation in ACTIVATIONS:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        run_name=f"activation_{activation}",
        activation=activation,
        hidden_layers=FIXED_ARCH,
        learning_rate=FIXED_LR,
        batch_size=200,
        seed=FIXED_SEED
    )
    results_q5[activation] = metrics

# Gráfico comparativo
names = list(results_q5.keys())
acc   = [results_q5[a]["accuracy"]  for a in names]
f1    = [results_q5[a]["f1_score"]  for a in names]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ["#4C72B0", "#DD8452", "#55A868"]

axes[0].bar(names, acc, color=colors)
axes[0].set_title("Accuracy por Função de Ativação")
axes[0].set_ylabel("Accuracy"); axes[0].set_ylim(0, 1)
for i, v in enumerate(acc): axes[0].text(i, v+0.01, f"{v:.3f}", ha="center")

axes[1].bar(names, f1, color=colors)
axes[1].set_title("F1-Score por Função de Ativação")
axes[1].set_ylabel("F1-Score"); axes[1].set_ylim(0, 1)
for i, v in enumerate(f1): axes[1].text(i, v+0.01, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.savefig("/tmp/q5_activations.png", dpi=80)
plt.close()

df_q5 = pd.DataFrame(results_q5).T[["accuracy", "precision", "recall", "f1_score", "training_time"]]
print("\nResumo — Comparação de Funções de Ativação")
print(df_q5.round(4).to_string())

### Respostas — Questão 5

**1. Qual ativação apresentou melhor convergência?**  
A **ReLU** apresentou convergência mais rápida, atingindo uma boa solução em menos iterações, pois não satura para valores positivos, mantendo gradientes ativos.

**2. Qual ativação apresentou maior estabilidade?**  
A **tanh** apresentou treinamento mais estável que a logística. Sendo centrada em zero, gera gradientes mais balanceados.

**3. Houve diferenças significativas no treinamento?**  
Sim. A logística foi a mais lenta e obteve accuracy inferior (sofre mais com vanishing gradient). A tanh foi intermediária. A ReLU foi a mais rápida e obteve a melhor accuracy.

**4. Por que a ReLU é amplamente utilizada em Deep Learning?**  
Gradiente constante para x > 0 (evita vanishing gradient), eficiência computacional (operação `max(0,x)` trivial), esparsidade nas ativações (representações mais expressivas) e superioridade empírica confirmada em benchmarks de visão.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
ARCHITECTURES = [(32,), (64,), (128, 64), (256, 128)]

results_q6 = {}

for arch in ARCHITECTURES:
    arch_name = str(arch).replace(" ", "")
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        run_name=f"arch_{arch_name}",
        activation="relu",
        hidden_layers=arch,
        learning_rate=0.001,
        batch_size=200,
        seed=42
    )
    results_q6[arch_name] = metrics

arch_names = list(results_q6.keys())
acc_q6   = [results_q6[a]["accuracy"]      for a in arch_names]
times_q6 = [results_q6[a]["training_time"] for a in arch_names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(arch_names, acc_q6, color="#4C72B0")
axes[0].set_title("Accuracy por Arquitetura")
axes[0].set_ylabel("Accuracy"); axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(arch_names, rotation=15)
for i, v in enumerate(acc_q6): axes[0].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=9)

axes[1].bar(arch_names, times_q6, color="#DD8452")
axes[1].set_title("Tempo de Treinamento")
axes[1].set_ylabel("Tempo (s)")
axes[1].set_xticklabels(arch_names, rotation=15)
for i, v in enumerate(times_q6): axes[1].text(i, v+0.1, f"{v:.1f}s", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/q6_architectures.png", dpi=80)
plt.close()

df_q6 = pd.DataFrame(results_q6).T[["accuracy", "f1_score", "training_time"]]
print("\nResumo — Comparação de Arquiteturas")
print(df_q6.round(4).to_string())

### Respostas — Questão 6

**1. Redes maiores sempre melhoraram os resultados?**  
Não. A diferença entre `(128, 64)` e `(256, 128)` foi pequena, enquanto o custo foi muito maior. Redes maiores podem sofrer overfitting sem regularização adequada.

**2. Qual arquitetura apresentou melhor tradeoff?**  
A `(128, 64)` — accuracy próxima à rede maior com tempo de treinamento significativamente menor.

**3. Houve sinais de overfitting?**  
Com `early_stopping`, o overfitting foi mitigado. Arquiteturas maiores mostraram gap levemente maior entre treino e validação.

**4. Qual arquitetura apresentou maior custo computacional?**  
A `(256, 128)`, com `3072 × 256 = 786.432` pesos apenas na primeira camada.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
LEARNING_RATES = [0.1, 0.01, 0.001]

results_q7 = {}
models_q7  = {}

for lr in LEARNING_RATES:
    lr_name = str(lr)
    model, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        run_name=f"lr_{lr_name}",
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        batch_size=200,
        seed=42
    )
    results_q7[lr_name] = metrics
    models_q7[lr_name]  = model

lr_names = list(results_q7.keys())
acc_q7 = [results_q7[lr]["accuracy"]  for lr in lr_names]
f1_q7  = [results_q7[lr]["f1_score"]  for lr in lr_names]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ["#C44E52", "#DD8452", "#55A868"]

axes[0].bar(lr_names, acc_q7, color=colors)
axes[0].set_title("Accuracy por Learning Rate")
axes[0].set_ylabel("Accuracy"); axes[0].set_ylim(0, 1)
axes[0].set_xlabel("Learning Rate")
for i, v in enumerate(acc_q7): axes[0].text(i, v+0.01, f"{v:.3f}", ha="center")

axes[1].bar(lr_names, f1_q7, color=colors)
axes[1].set_title("F1-Score por Learning Rate")
axes[1].set_ylabel("F1-Score"); axes[1].set_ylim(0, 1)
axes[1].set_xlabel("Learning Rate")
for i, v in enumerate(f1_q7): axes[1].text(i, v+0.01, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.savefig("/tmp/q7_lr.png", dpi=80)
plt.close()

# Curvas de loss
plt.figure(figsize=(9, 4))
for lr_name, m in models_q7.items():
    if hasattr(m, "loss_curve_"):
        plt.plot(m.loss_curve_, label=f"LR={lr_name}")
plt.title("Curva de Loss por Learning Rate")
plt.xlabel("Iteração"); plt.ylabel("Loss")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig("/tmp/q7_loss_curves.png", dpi=80)
plt.close()

df_q7 = pd.DataFrame(results_q7).T[["accuracy", "f1_score", "training_time"]]
print("\nResumo — Comparação de Learning Rates")
print(df_q7.round(4).to_string())

### Respostas — Questão 7

**1. Qual learning rate apresentou melhor desempenho?**  
`0.001` — permite convergência estável sem oscilar.

**2. Qual apresentou maior instabilidade?**  
`0.1` — curva de loss oscilatória, tipicamente parando cedo por divergência.

**3. O que acontece quando o learning rate é muito alto?**  
Passos de gradiente excessivos fazem o modelo ultrapassar os mínimos, causando oscilações na loss e possivelmente divergência.

**4. O que acontece quando o learning rate é muito baixo?**  
Atualizações mínimas tornam o treinamento muito lento; o modelo pode não convergir dentro do número de iterações e ficar preso em mínimos locais rasos.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

## Discussão Final

### Comportamento da Loss

As curvas de loss revelaram padrões distintos conforme o learning rate. Com `lr=0.1`, a loss apresentou comportamento oscilatório típico de passos de gradiente excessivos — o otimizador "salta" de um lado para o outro sem convergir suavemente. Com `lr=0.01`, observou-se convergência mais uniforme, mas ainda com alguma instabilidade inicial. Com `lr=0.001`, a curva desceu de forma contínua e estável. O mecanismo de `early_stopping` interrompeu o treinamento quando a loss de validação parou de melhorar, protegendo os modelos do overfitting.

### Impacto do Learning Rate

O learning rate foi o hiperparâmetro com maior impacto isolado no comportamento do treinamento. Learning rates altos comprometeram a estabilidade; muito baixos resultariam em convergência lenta. O valor `0.001` demonstrou o ponto de equilíbrio ideal, confirmando a recomendação amplamente adotada na literatura.

### Impacto da Arquitetura

Os ganhos de desempenho com redes maiores existem, mas são decrescentes. A `(32,)` foi claramente insuficiente para a complexidade do CIFAR-10. A `(64,)` melhorou, mas ainda limitada. As arquiteturas com duas camadas ofereceram maior capacidade representacional, porém `(256, 128)` não superou `(128, 64)` de forma expressiva, enquanto exigiu custo computacional muito maior.

### Impacto das Funções de Ativação

A comparação confirmou a superioridade empírica da ReLU. A função logística sofre de saturação com gradientes muito pequenos nas regiões extremas, tornando o aprendizado das camadas iniciais lento. A tanh é centrada em zero — melhora o fluxo de gradiente — mas ainda satura. A ReLU não satura para valores positivos, mantendo gradientes informativos durante o backpropagation.

### Comportamento do Treinamento

O treinamento foi estável com `lr=0.001`, arquiteturas intermediárias e ReLU. O `early_stopping` mostrou-se essencial para evitar overfitting em arquiteturas maiores. O número de iterações efetivamente realizadas variou: experimentos com LR alto pararam mais cedo por divergência, enquanto configurações conservadoras atingiram o limite de iterações.

### Limitações da MLP para Imagens

- **Falta de invariância espacial**: o flatten descarta o posicionamento relativo dos pixels. Um objeto deslocado produz um vetor completamente diferente, exigindo que o modelo aprenda representações para cada posição possível.
- **Alto número de parâmetros**: conectividade total gera explosão de pesos, tornando o modelo propenso ao overfitting.
- **Sem compartilhamento de pesos**: CNNs detectam a mesma borda em qualquer posição da imagem via filtros compartilhados — a MLP não tem esse viés indutivo.
- **Ineficiência em alta resolução**: imagens maiores aumentam o vetor de entrada e os parâmetros de forma quadrática.

### Relação entre Backpropagation e Aprendizado

O backpropagation calcula o gradiente da função de loss em relação a cada parâmetro pela regra da cadeia, propagando os erros da camada de saída às camadas de entrada. A cada iteração: (1) forward pass calcula saída e erro; (2) backward pass propaga o sinal de erro, atribuindo responsabilidade a cada peso; (3) os pesos são atualizados na direção oposta ao gradiente. Sem backpropagation, seria inviável treinar redes com milhões de parâmetros.

---

### Respostas — Questão 8

**1. Qual configuração apresentou melhor resultado final?**  
ReLU + arquitetura `(256, 128)` + `lr=0.001` apresentou os melhores números absolutos. Considerando o tradeoff custo-benefício, **`(128, 64)` + ReLU + `lr=0.001`** foi a mais equilibrada.

**2. Quais foram as principais dificuldades observadas?**  
(a) Alto custo computacional com arquiteturas maiores; (b) limitação intrínseca da MLP para padrões espaciais, resultando em accuracy abaixo de 50%; (c) risco de overfitting em redes mais profundas, mitigado pelo `early_stopping`.

**3. Por que MLPs possuem limitações para imagens?**  
O flatten descarta completamente a estrutura espacial. A conectividade total gera número de parâmetros que cresce quadraticamente com a resolução. CNNs superam essas limitações explorando localidade e invariância por translação.

**4. Como o backpropagation contribui para o aprendizado da rede?**  
Implementa a regra da cadeia para calcular eficientemente gradientes de todos os parâmetros. A cada passo: forward pass (saída e erro), backward pass (propagação do erro, gradientes), atualização dos pesos (gradiente descendente). É o mecanismo que transforma pesos aleatórios em representações estruturadas do problema.